In [1]:
# AI Fake News Detection System
## Data Preprocessing

In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import string
import warnings

warnings.filterwarnings("ignore")

In [3]:
# Display pandas settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## Load Dataset

In [4]:
# Load Fake News dataset
fake_df = pd.read_csv("../data/Fake.csv")

# Load Real News dataset
true_df = pd.read_csv("../data/True.csv")

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [5]:
# Add labels to both datasets
fake_df["label"] = 0
true_df["label"] = 1

In [6]:
# Merge both datasets
df = pd.concat([fake_df, true_df], ignore_index=True)

print("Datasets merged successfully.")

Datasets merged successfully.


In [7]:
# Display dataset shape

print(df.shape)

(44898, 5)


In [8]:
# Display first five records

df.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing,"Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and the very dishonest fake news media. The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year, President Angry Pants tweeted. 2018 will be a great year for America! As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year. 2018 will be a great year for America! Donald J. Trump (@realDonaldTrump) December 31, 2017Trump s tweet went down about as welll as you d expect.What kind of president sends a New Year s greeting like this despicable, petty, infantile gibberish? Only Trump! His lack of decency won t even allow him to rise above the gutter long enough to wish the American citizens a happy new year! Bishop Talbert Swan (@TalbertSwan) December 31, 2017no one likes you Calvin (@calvinstowell) December 31, 2017Your impeachment would make 2018 a great year for America, but I ll also accept regaining control of Congress. Miranda Yaver (@mirandayaver) December 31, 2017Do you hear yourself talk? When you have to include that many people that hate you you have to wonder? Why do the they all hate me? Alan Sandoval (@AlanSandoval13) December 31, 2017Who uses the word Haters in a New Years wish?? Marlene (@marlene399) December 31, 2017You can t just say happy new year? Koren pollitt (@Korencarpenter) December 31, 2017Here s Trump s New Year s Eve tweet from 2016.Happy New Year to all, including to my many enemies and those who have fought me and lost so badly they just don t know what to do. Love! Donald J. Trump (@realDonaldTrump) December 31, 2016This is nothing new for Trump. He s been doing this for years.Trump has directed messages to his enemies and haters for New Year s, Easter, Thanksgiving, and the anniversary of 9/11. pic.twitter.com/4FPAe2KypA Daniel Dale (@ddale8) December 31, 2017Trump s holiday tweets are clearly not presidential.How long did he work at Hallmark before becoming President? Steven Goodine (@SGoodine) December 31, 2017He s always been like this . . . the only difference is that in the last few years, his filter has been breaking down. Roy Schulze (@thbthttt) December 31, 2017Who, apart from a teenager uses the term haters? Wendy (@WendyWhistles) December 31, 2017he s a fucking 5 year old Who Knows (@rainyday80) December 31, 2017So, to all the people who voted for this a hole thinking he would change once he got into power, you were wrong! 70-year-old men don t change and now he s a year older.Photo by Andrew Burton/Getty Images.",News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian Collusion Investigation,"House Intelligence Committee Chairman Devin Nunes is going to have a bad day. He s been under the assumption, like many of us, that the Christopher Steele-dossier was what prompted the Russia investigation so he s been lashing out at the Department of Justice and the FBI in order to protect Trump. As it happens, the dossier is not what started the investigation, according to documents obtained by the New York Times.Former Trump campaign adviser George Papadopoulos was drunk in a wine bar when he revealed knowledge of Russian opposition research on Hillary Clinton.On top of that, Papadopoulos wasn t just a covfefe boy for Trump, as his administration has alleged. He had a much larger role, but none so damning as being a drunken fool in a wine bar. Coffee boys don t help to arrange a New York meeting between Trump and President Abdel Fattah el-Sisi of Egypt two months before the election. It was known b

## Text Preprocessing

In [9]:
# Display dataset information before preprocessing

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    44898 non-null  str  
 1   text     44898 non-null  str  
 2   subject  44898 non-null  str  
 3   date     44898 non-null  str  
 4   label    44898 non-null  int64
dtypes: int64(1), str(4)
memory usage: 112.3 MB


In [10]:
# Check missing values before preprocessing

df.isnull().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

## Convert Text to Lowercase

In [11]:
# Convert news text to lowercase
df["text"] = df["text"].str.lower()

print("Text converted to lowercase successfully.")

Text converted to lowercase successfully.


In [12]:
# Display first news article after lowercase conversion

df["text"].head(1)

0    donald trump just couldn t wish all americans a happy new year and leave it at that. instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  the former reality show star had just one job to do and he couldn t do it. as our country rapidly grows stronger and smarter, i want to wish all of my friends, supporters, enemies, haters, and even the very dishonest fake news media, a happy and healthy new year,  president angry pants tweeted.  2018 will be a great year for america! as our country rapidly grows stronger and smarter, i want to wish all of my friends, supporters, enemies, haters, and even the very dishonest fake news media, a happy and healthy new year. 2018 will be a great year for america!  donald j. trump (@realdonaldtrump) december 31, 2017trump s tweet went down about as welll as you d expect.what kind of president sends a new year s greeting like this despicable, petty, infantile gibberish? only trump! his lack of decency won 

## Convert News Title to Lowercase

In [13]:
# Convert news title to lowercase
df["title"] = df["title"].str.lower()

print("Title converted to lowercase successfully.")

Title converted to lowercase successfully.


In [14]:
# Display first news title after lowercase conversion

df["title"].head(1)

0     donald trump sends out embarrassing new year’s eve message; this is disturbing
Name: title, dtype: str

## Remove HTML Tags

In [15]:
# Import BeautifulSoup
from bs4 import BeautifulSoup

In [16]:
# Create a function to remove HTML tags
def remove_html(text):
    return BeautifulSoup(str(text), "html.parser").get_text()

In [17]:
# Remove HTML tags from news text
df["text"] = df["text"].apply(remove_html)

print("HTML tags removed successfully from text.")

HTML tags removed successfully from text.


In [18]:
# Remove HTML tags from news title

df["title"] = df["title"].apply(remove_html)

print("HTML tags removed successfully from title.")

HTML tags removed successfully from title.


In [19]:
# Display the first news article after HTML tag removal

df["text"].head(1)

0    donald trump just couldn t wish all americans a happy new year and leave it at that. instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  the former reality show star had just one job to do and he couldn t do it. as our country rapidly grows stronger and smarter, i want to wish all of my friends, supporters, enemies, haters, and even the very dishonest fake news media, a happy and healthy new year,  president angry pants tweeted.  2018 will be a great year for america! as our country rapidly grows stronger and smarter, i want to wish all of my friends, supporters, enemies, haters, and even the very dishonest fake news media, a happy and healthy new year. 2018 will be a great year for america!  donald j. trump (@realdonaldtrump) december 31, 2017trump s tweet went down about as welll as you d expect.what kind of president sends a new year s greeting like this despicable, petty, infantile gibberish? only trump! his lack of decency won 

## Remove URLs

In [20]:
# Create a function to remove URLs
def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', str(text))

In [21]:
# Remove URLs from news text
df["text"] = df["text"].apply(remove_url)

print("URLs removed successfully from text.")

URLs removed successfully from text.


In [22]:
# Remove URLs from news title
df["title"] = df["title"].apply(remove_url)

print("URLs removed successfully from title.")

URLs removed successfully from title.


In [23]:
# Display the first news article after URL removal

df["text"].head(1)

0    donald trump just couldn t wish all americans a happy new year and leave it at that. instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  the former reality show star had just one job to do and he couldn t do it. as our country rapidly grows stronger and smarter, i want to wish all of my friends, supporters, enemies, haters, and even the very dishonest fake news media, a happy and healthy new year,  president angry pants tweeted.  2018 will be a great year for america! as our country rapidly grows stronger and smarter, i want to wish all of my friends, supporters, enemies, haters, and even the very dishonest fake news media, a happy and healthy new year. 2018 will be a great year for america!  donald j. trump (@realdonaldtrump) december 31, 2017trump s tweet went down about as welll as you d expect.what kind of president sends a new year s greeting like this despicable, petty, infantile gibberish? only trump! his lack of decency won 

## Remove Punctuation

In [24]:
# Create a function to remove punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))

In [25]:
# Remove punctuation from news text
df["text"] = df["text"].apply(remove_punctuation)

print("Punctuation removed successfully from text.")

Punctuation removed successfully from text.


In [26]:
# Remove punctuation from news title
df["title"] = df["title"].apply(remove_punctuation)

print("Punctuation removed successfully from title.")

Punctuation removed successfully from title.


In [27]:
# Display the first news article after punctuation removal

df["text"].head(1)

0    donald trump just couldn t wish all americans a happy new year and leave it at that instead he had to give a shout out to his enemies haters and  the very dishonest fake news media  the former reality show star had just one job to do and he couldn t do it as our country rapidly grows stronger and smarter i want to wish all of my friends supporters enemies haters and even the very dishonest fake news media a happy and healthy new year  president angry pants tweeted  2018 will be a great year for america as our country rapidly grows stronger and smarter i want to wish all of my friends supporters enemies haters and even the very dishonest fake news media a happy and healthy new year 2018 will be a great year for america  donald j trump realdonaldtrump december 31 2017trump s tweet went down about as welll as you d expectwhat kind of president sends a new year s greeting like this despicable petty infantile gibberish only trump his lack of decency won t even allow him to rise above t

## Remove Numbers

In [28]:
# Create a function to remove numbers
def remove_numbers(text):
    return re.sub(r"\d+", "", str(text))

In [29]:
# Remove numbers from news text
df["text"] = df["text"].apply(remove_numbers)

print("Numbers removed successfully from text.")

Numbers removed successfully from text.


In [30]:
# Remove numbers from news title
df["title"] = df["title"].apply(remove_numbers)

print("Numbers removed successfully from title.")

Numbers removed successfully from title.


In [31]:
# Display the first news article after number removal

df["text"].head(1)

0    donald trump just couldn t wish all americans a happy new year and leave it at that instead he had to give a shout out to his enemies haters and  the very dishonest fake news media  the former reality show star had just one job to do and he couldn t do it as our country rapidly grows stronger and smarter i want to wish all of my friends supporters enemies haters and even the very dishonest fake news media a happy and healthy new year  president angry pants tweeted   will be a great year for america as our country rapidly grows stronger and smarter i want to wish all of my friends supporters enemies haters and even the very dishonest fake news media a happy and healthy new year  will be a great year for america  donald j trump realdonaldtrump december  trump s tweet went down about as welll as you d expectwhat kind of president sends a new year s greeting like this despicable petty infantile gibberish only trump his lack of decency won t even allow him to rise above the gutter long

## Remove Special Characters

In [32]:
# Create a function to remove special characters

def remove_special_characters(text):
    return re.sub(r"[^a-zA-Z\s]", "", str(text))

In [33]:
# Remove special characters from news text

df["text"] = df["text"].apply(remove_special_characters)

print("Special characters removed successfully from text.")

Special characters removed successfully from text.


In [34]:
# Remove special characters from news title

df["title"] = df["title"].apply(remove_special_characters)

print("Special characters removed successfully from title.")

Special characters removed successfully from title.


In [35]:
# Display the first news article after removing special characters

df["text"].head(1)

0    donald trump just couldn t wish all americans a happy new year and leave it at that instead he had to give a shout out to his enemies haters and  the very dishonest fake news media  the former reality show star had just one job to do and he couldn t do it as our country rapidly grows stronger and smarter i want to wish all of my friends supporters enemies haters and even the very dishonest fake news media a happy and healthy new year  president angry pants tweeted   will be a great year for america as our country rapidly grows stronger and smarter i want to wish all of my friends supporters enemies haters and even the very dishonest fake news media a happy and healthy new year  will be a great year for america  donald j trump realdonaldtrump december  trump s tweet went down about as welll as you d expectwhat kind of president sends a new year s greeting like this despicable petty infantile gibberish only trump his lack of decency won t even allow him to rise above the gutter long

## Remove Extra Spaces

In [36]:
# Create a function to remove extra spaces

def remove_extra_spaces(text):
    return re.sub(r"\s+", " ", str(text)).strip()

In [37]:
# Remove extra spaces from news text

df["text"] = df["text"].apply(remove_extra_spaces)

print("Extra spaces removed successfully from text.")

Extra spaces removed successfully from text.


In [38]:
# Remove extra spaces from news title

df["title"] = df["title"].apply(remove_extra_spaces)

print("Extra spaces removed successfully from title.")

Extra spaces removed successfully from title.


In [39]:
# Display the first news article after removing extra spaces

df["text"].head(1)

0    donald trump just couldn t wish all americans a happy new year and leave it at that instead he had to give a shout out to his enemies haters and the very dishonest fake news media the former reality show star had just one job to do and he couldn t do it as our country rapidly grows stronger and smarter i want to wish all of my friends supporters enemies haters and even the very dishonest fake news media a happy and healthy new year president angry pants tweeted will be a great year for america as our country rapidly grows stronger and smarter i want to wish all of my friends supporters enemies haters and even the very dishonest fake news media a happy and healthy new year will be a great year for america donald j trump realdonaldtrump december trump s tweet went down about as welll as you d expectwhat kind of president sends a new year s greeting like this despicable petty infantile gibberish only trump his lack of decency won t even allow him to rise above the gutter long enough 

## Tokenization

In [40]:
# Import NLTK
import nltk

In [41]:
# Download punkt tokenizer

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to C:\Users\MY
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\MY
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [42]:

# Import word_tokenize
from nltk.tokenize import word_tokenize

In [43]:
# Tokenize the news text
df["tokens"] = df["text"].apply(word_tokenize)

print("Text tokenization completed successfully.")

Text tokenization completed successfully.


In [44]:
# Display tokens of the first news article

df["tokens"].head(1)

0    [donald, trump, just, couldn, t, wish, all, americans, a, happy, new, year, and, leave, it, at, that, instead, he, had, to, give, a, shout, out, to, his, enemies, haters, and, the, very, dishonest, fake, news, media, the, former, reality, show, star, had, just, one, job, to, do, and, he, couldn, t, do, it, as, our, country, rapidly, grows, stronger, and, smarter, i, want, to, wish, all, of, my, friends, supporters, enemies, haters, and, even, the, very, dishonest, fake, news, media, a, happy, and, healthy, new, year, president, angry, pants, tweeted, will, be, a, great, year, for, america, as, our, country, ...]
Name: tokens, dtype: object

## Stopword Removal

In [45]:
# Download stopwords

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to C:\Users\MY
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [46]:
# Import stopwords
from nltk.corpus import stopwords

In [47]:
# Store English stopwords
stop_words = set(stopwords.words("english"))

In [48]:
# Remove stopwords from tokens
df["tokens"] = df["tokens"].apply(
    lambda words: [word for word in words if word not in stop_words]
)
print("Stopwords removed successfully.")

Stopwords removed successfully.


In [49]:
# Display tokens after stopword removal

df["tokens"].head(1)

0    [donald, trump, wish, americans, happy, new, year, leave, instead, give, shout, enemies, haters, dishonest, fake, news, media, former, reality, show, star, one, job, country, rapidly, grows, stronger, smarter, want, wish, friends, supporters, enemies, haters, even, dishonest, fake, news, media, happy, healthy, new, year, president, angry, pants, tweeted, great, year, america, country, rapidly, grows, stronger, smarter, want, wish, friends, supporters, enemies, haters, even, dishonest, fake, news, media, happy, healthy, new, year, great, year, america, donald, j, trump, realdonaldtrump, december, trump, tweet, went, welll, expectwhat, kind, president, sends, new, year, greeting, like, despicable, petty, infantile, gibberish, trump, lack, decency, even, allow, rise, ...]
Name: tokens, dtype: object

## Lemmatization using spaCy

In [50]:
# Import spaCy
import spacy

In [51]:
# Load English language model
nlp = spacy.load("en_core_web_sm")

print("spaCy model loaded successfully.")

spaCy model loaded successfully.


In [52]:
# Create a function for Lemmatization

def lemmatize_text(tokens):
    doc = nlp(" ".join(tokens))
    return [token.lemma_ for token in doc]

In [53]:
tokens = ["this", "is", "a", "test"]
print(lemmatize_text(tokens))

['this', 'be', 'a', 'test']


In [54]:
# Apply Lemmatization using spaCy nlp.pipe 
docs = nlp.pipe(
    (" ".join(tokens) for tokens in df["tokens"]),
    batch_size=100
)

df["tokens"] = [
    [token.lemma_ for token in doc]
    for doc in docs
]

print("Lemmatization completed successfully.")

# Apply lemmatization
#df["tokens"] = df["tokens"].apply(lemmatize_text)

#print("Lemmatization completed successfully.")

Lemmatization completed successfully.


In [55]:
# Display lemmatized tokens
df["tokens"].head(1)

0    [donald, trump, wish, americans, happy, new, year, leave, instead, give, shout, enemy, hater, dishonest, fake, news, medium, former, reality, show, star, one, job, country, rapidly, grow, strong, smart, want, wish, friend, supporter, enemy, hater, even, dishonest, fake, news, medium, happy, healthy, new, year, president, angry, pant, tweet, great, year, america, country, rapidly, grow, strong, smart, want, wish, friend, supporter, enemy, hater, even, dishonest, fake, news, medium, happy, healthy, new, year, great, year, america, donald, j, trump, realdonaldtrump, december, trump, tweet, go, welll, expectwhat, kind, president, send, new, year, greeting, like, despicable, petty, infantile, gibberish, trump, lack, decency, even, allow, rise, ...]
Name: tokens, dtype: object

## Create Clean Text

In [56]:
# Join tokens to create clean text
df["clean_text"] = df["tokens"].apply(lambda words: " ".join(words))

print("Clean text created successfully.")

Clean text created successfully.


In [57]:
# Compare original and cleaned text
df[["text", "clean_text"]].head()

,text,clean_text
0,donald trump just couldn t wish all americans a happy new year and leave it at that instead he had to give a shout out to his enemies haters and the very dishonest fake news media the former reality show star had just one job to do and he couldn t do it as our country rapidly grows stronger and smarter i want to wish all of my friends supporters enemies haters and even the very dishonest fake news media a happy and healthy new year president angry pants tweeted will be a great year for america as our country rapidly grows stronger and smarter i want to wish all of my friends supporters enemies haters and even the very dishonest fake news media a happy and healthy new year will be a great year for america donald j trump realdonaldtrump december trump s tweet went down about as welll as you d expectwhat kind of president sends a new year s greeting like this despicable petty infantile gibberish only trump his lack of decency won t even allow him to rise above the gutter long enough to wish the american citizens a happy new year bishop talbert swan talbertswan december no one likes you calvin calvinstowell december your impeachment would make a great year for america but i ll also accept regaining control of congress miranda yaver mirandayaver december do you hear yourself talk when you have to include that many people that hate you you have to wonder why do the they all hate me alan sandoval alansandoval december who uses the word haters in a new years wish marlene marlene december you can t just say happy new year koren pollitt korencarpenter december here s trump s new year s eve tweet from happy new year to all including to my many enemies and those who have fought me and lost so badly they just don t know what to do love donald j trump realdonaldtrump december this is nothing new for trump he s been doing this for yearstrump has directed messages to his enemies and haters for new year s easter thanksgiving and the anniversary of pictwittercomfpaekypa daniel dale ddale december trump s holiday tweets are clearly not presidentialhow long did he work at hallmark before becoming president steven goodine sgoodine december he s always been like this the only difference is that in the last few years his filter has been breaking down roy schulze thbthttt december who apart from a teenager uses the term haters wendy wendywhistles december he s a fucking year old who knows rainyday december so to all the people who voted for this a hole thinking he would change once he got into power you were wrong yearold men don t change and now he s a year olderphoto by andrew burtongetty images,donald trump wish americans happy new year leave instead give shout enemy hater dishonest fake news medium former reality show star one job country rapidly grow strong smart want wish friend supporter enemy hater even dishonest fake news medium happy healthy new year president angry pant tweet great year america country rapidly grow strong smart want wish friend supporter enemy hater even dishonest fake news medium happy healthy new year great year america donald j trump realdonaldtrump december trump tweet go welll expectwhat kind president send new year greeting like despicable petty infantile gibberish trump lack decency even allow rise gutter long enough wish american citizen happy new year bishop talbert swan talbertswan december one like calvin calvinstowell december impeachment would make great year america also accept regain control congress miranda yaver mirandayaver december hear talk include many people hate wonder hate alan sandoval alansandoval december use word hater new year wish marlene marlene december say happy new year koren pollitt korencarpenter december trump new year eve tweet happy new year include many enemy fight lose badly know love donald j trump realdonaldtrump december nothing new trump yearstrump direct message enemy hater new year easter thanksgive anniversary pictwittercomfpaekypa daniel dale ddale decembe

In [58]:
# Display the first cleaned news article
df["clean_text"].head(1)

0    donald trump wish americans happy new year leave instead give shout enemy hater dishonest fake news medium former reality show star one job country rapidly grow strong smart want wish friend supporter enemy hater even dishonest fake news medium happy healthy new year president angry pant tweet great year america country rapidly grow strong smart want wish friend supporter enemy hater even dishonest fake news medium happy healthy new year great year america donald j trump realdonaldtrump december trump tweet go welll expectwhat kind president send new year greeting like despicable petty infantile gibberish trump lack decency even allow rise gutter long enough wish american citizen happy new year bishop talbert swan talbertswan december one like calvin calvinstowell december impeachment would make great year america also accept regain control congress miranda yaver mirandayaver december hear talk include many people hate wonder hate alan sandoval alansandoval december use word hater

## Final Dataset Validation

In [59]:
# Display dataset information after preprocessing
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   title       44898 non-null  str   
 1   text        44898 non-null  str   
 2   subject     44898 non-null  str   
 3   date        44898 non-null  str   
 4   label       44898 non-null  int64 
 5   tokens      44898 non-null  object
 6   clean_text  44898 non-null  str   
dtypes: int64(1), object(1), str(5)
memory usage: 178.8+ MB


In [60]:
# Check missing values after preprocessing
df.isnull().sum()

title         0
text          0
subject       0
date          0
label         0
tokens        0
clean_text    0
dtype: int64

In [61]:
# Display the final dataset shape
print("Final Dataset Shape:", df.shape)

Final Dataset Shape: (44898, 7)


In [62]:
# Display sample records
df[["title", "clean_text", "label"]].sample(5, random_state=42)

,title,clean_text,label
22216,ben stein calls out th circuit court committed a coup dtat against the constitution,st century wire say ben stein reputable professor pepperdine university also hollywood fame appear tv show film ferris bueller day make provocative statement judge jeanine pirro show recently discuss halt impose president trump executive order travel stein refer judgement th circuit court washington state coup tat executive branch constitution stein go call judge seattle political puppet judiciary political pawn watch interview complete statement note stark contrast rhetoric leftist medium pundit neglect note court ever block presidential order immigration past discuss legal efficacy halt actual text executive orderread trump news st century wire trump filessupport work subscribe become member wiretv,0
27917,trump drops steve bannon from national security council,washington reuters us president donald trump remove chief strategist steve bannon national security council wednesday reverse controversial decision early year give political adviser unprecedented role security discussion trump overhaul nsc confirm white house official also elevate general joseph dunford chairman joint chiefs staff dan coats director national intelligence head we intelligence agency official say change move nsc back core function suppose also appear mark victory national security adviser hr mcmaster tell national security expert feel battle death bannon other white house staff vice president mike pence say bannon would continue play important role policy play shakeup routine natural evolution ensure national security council organize way well serve president resolve make difficult decision pence say fox news bannon say statement succeed return nsc back traditional role coordinate foreign policy rather run cited president barack obamas national security adviser susan rice advocate change susan rice operationalized nsc last administration put nsc ensure deoperationalized general mcmaster nsc back proper function say trump white house team grapple infighting intrigue hobble young presidency recent day several senior us foreign policy national security official say mechanism shape trump administration response press challenge syria north korea iran still place critic bannon role nsc say give much weight decisionmake someone lack foreign policy expertise bannon chief executive trump presidential campaign month lead election november respect represent trump america first nationalistic voice help fuel antiwashington fervor push president part way times mainstream republicans join trump administration bannon head breitbart news rightwing website we representative adam schiff rank democrat house representatives intelligence committee call shift nsc positive step help mcmaster gain control body politicize bannon involvement administration policy north korea china russia syria continue drift hope shakeup bring level strategic vision body say bannon removal nsc potential setback sphere influence trump white house voice major decision trump confidant say bannon remain influential ever still involve everything still full confidence president fair much stuff confidant say speak condition anonymity white house official say bannon long need nsc departure trump first national security adviser michael flynn flynn force resign feb contact russias ambassador united states sergei kislyak prior trump take office jan official say bannon place nsc originally check flynn attend one nscs regular meeting official dismiss question power struggle bannon mcmaster say share world view however two current national security official reject white house explanation note two month pass since flynns departure mcmaster say speak condition anonymity also duel bannon other direct access trump future deputy national security adviser kt mcfarland former fox news commentator intelligence director ezra cohenwatnick flynn appointee staffing decision trump prepare first facetoface meet

## Save Cleaned Dataset

In [63]:
# Save the cleaned dataset
df.to_csv("cleaned_data.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


In [64]:
# Verify the saved dataset
cleaned_df = pd.read_csv("cleaned_data.csv")

print(cleaned_df.shape)

(44898, 7)


## Data Preprocessing Summary

### Key Preprocessing Steps

- Loaded and merged Fake News and Real News datasets.
- Converted text and titles to lowercase.
- Removed HTML tags and URLs.
- Removed punctuation, numbers, and special characters.
- Removed extra spaces.
- Performed tokenization using NLTK.
- Removed English stopwords.
- Applied lemmatization using spaCy.
- Created a clean text column for NLP tasks.
- Saved the cleaned dataset for feature extraction and model training.

In [65]:
df["tokens"].head(1)

0    [donald, trump, wish, americans, happy, new, year, leave, instead, give, shout, enemy, hater, dishonest, fake, news, medium, former, reality, show, star, one, job, country, rapidly, grow, strong, smart, want, wish, friend, supporter, enemy, hater, even, dishonest, fake, news, medium, happy, healthy, new, year, president, angry, pant, tweet, great, year, america, country, rapidly, grow, strong, smart, want, wish, friend, supporter, enemy, hater, even, dishonest, fake, news, medium, happy, healthy, new, year, great, year, america, donald, j, trump, realdonaldtrump, december, trump, tweet, go, welll, expectwhat, kind, president, send, new, year, greeting, like, despicable, petty, infantile, gibberish, trump, lack, decency, even, allow, rise, ...]
Name: tokens, dtype: object